# Notebook 3: Fraud Model Training (XGBoost)

Trains an XGBoost fraud classifier on 12M transactions with 147 features. Training data is generated directly from the `FRAUD_TRANSACTIONS` table — no Dynamic Tables or Feature Store required for training.

---

### Design Choices

| Decision | Choice | Rationale |
|---|---|---|
| Warehouse | FRAUD_OFS_TRAIN_WH (SP-Opt MEDIUM, 6 cr/hr) | 256GB dedicated RAM. 12M x 147 features fits comfortably. |
| Why not Standard XLARGE | SP-Opt MEDIUM is 62% cheaper AND more memory | Standard XLARGE = 16 cr/hr, ~80GB usable vs 256GB dedicated |
| tree_method | 'hist' | 5-10x faster than 'exact' at this scale, minimal accuracy loss |
| scale_pos_weight | 2000 (inverse fraud rate) | Handles 0.05% imbalance without memory-expensive oversampling |
| Evaluation metric | AUC-PR | ROC-AUC is misleading at 0.05% — a model predicting "never fraud" gets 99.95% accuracy |
| MAX_CONCURRENCY_LEVEL | 1 | Training gets exclusive access to all 256GB RAM |

In [ ]:
from snowflake.snowpark.context import get_active_session

# snowflake.ml.modeling.xgboost wraps the open-source XGBoost library so that training
# runs natively inside Snowflake on a Snowpark-Optimized warehouse (dedicated RAM/CPU),
# rather than pulling data to a local machine. No data ever leaves Snowflake.
from snowflake.ml.modeling.xgboost import XGBClassifier

# Registry is the Snowflake Model Registry — a versioned store for trained ML models.
# It handles serialisation, schema inference, and model promotion (DEV → STAGING → PROD).
from snowflake.ml.registry import Registry
from snowflake.snowpark.functions import col, lit, when, datediff, current_timestamp
from snowflake.snowpark.functions import hour, dayofweek
import time

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA ML').collect()

# Resume the Snowpark-Optimized MEDIUM warehouse if it auto-suspended.
# Snowpark-Optimized warehouses have dedicated memory (256GB on MEDIUM) vs shared on Standard.
# This is important for XGBoost — it needs to hold the full 1M-row training set in RAM.
session.sql('ALTER WAREHOUSE FRAUD_OFS_TRAIN_WH RESUME IF SUSPENDED').collect()
session.sql('USE WAREHOUSE FRAUD_OFS_TRAIN_WH').collect()
print('Context: FRAUD_DEMO_DEV.ML, Snowpark-Optimized MEDIUM warehouse')

## 3.1 Generate Training Dataset

Training features are computed directly from the transactions table using window aggregations. No Dynamic Tables or Feature Store needed for training — the Online FS is a serving layer only.

This approach eliminates the 24/7 DT warehouse cost entirely.

In [ ]:
print('Generating training dataset...')
import time
start = time.time()

# Training features use point-in-time correct self-joins on the transactions table.
# For each row t, we join back to count all PRIOR transactions in each time window.
# This produces ground-truth feature values for every row at the moment it occurred.
#
# Entities: customer (5 windows) + customer profile join.
# The scoring service fetches more features from OFS (merchant/DPAN/IP velocity),
# but the model only uses the feature_cols it was trained on — those map to customer
# velocity + profile + transaction attributes + derived ratios.
#
# SAMPLE (1000000 ROWS) keeps runtime manageable on the training warehouse.

training_df = session.sql("""
    SELECT
        -- Identifiers (excluded from model features in cell 5)
        t.CUSTOMER_ID, t.MERCHANT_ID, t.WALLET_DPAN, t.IP_ADDRESS,
        t.TRANSACTION_TS, t.IS_FRAUD,

        -- Transaction attributes
        t.PURCHASE_AMOUNT,
        CASE WHEN t.MERCHANT_COUNTRY = 'GBR' THEN 1.0 ELSE 0.0 END AS IS_GBR,
        t.AUTHENTICATED_3DS_CHALLENGE_FLAG,
        t.IS_MERCHANT_INITIATED_PURCHASE,

        -- Time features (circadian fraud patterns)
        HOUR(t.TRANSACTION_TS)                                          AS HOUR_OF_DAY,
        DAYOFWEEK(t.TRANSACTION_TS)                                     AS DAY_OF_WEEK,
        CASE WHEN DAYOFWEEK(t.TRANSACTION_TS) IN (0,6) THEN 1 ELSE 0 END AS IS_WEEKEND,
        CASE WHEN HOUR(t.TRANSACTION_TS) < 5 THEN 1 ELSE 0 END          AS IS_NIGHT,

        -- Customer velocity: point-in-time correct via self-joins
        -- h1 = prior transactions for same customer in last 1 hour
        COUNT(h1.TRANSACTION_TS)                    AS PURCHASES_NUM_L1H,
        COALESCE(SUM(h1.PURCHASE_AMOUNT), 0)        AS PURCHASES_AMT_L1H,
        COALESCE(MAX(h1.PURCHASE_AMOUNT), 0)        AS PURCHASES_MAX_L1H,
        COUNT(DISTINCT h1.MERCHANT_ID)              AS DISTINCT_MERCHANTS_L1H,
        -- h6 = 6-hour window
        COUNT(h6.TRANSACTION_TS)                    AS PURCHASES_NUM_L6H,
        COALESCE(SUM(h6.PURCHASE_AMOUNT), 0)        AS PURCHASES_AMT_L6H,
        COUNT(DISTINCT h6.MERCHANT_ID)              AS DISTINCT_MERCHANTS_L6H,
        -- h24 = 24-hour window
        COUNT(h24.TRANSACTION_TS)                   AS PURCHASES_NUM_L24H,
        COALESCE(SUM(h24.PURCHASE_AMOUNT), 0)       AS PURCHASES_AMT_L24H,
        COALESCE(MAX(h24.PURCHASE_AMOUNT), 0)       AS PURCHASES_MAX_L24H,
        COUNT(DISTINCT h24.MERCHANT_ID)             AS DISTINCT_MERCHANTS_L24H,
        COUNT(DISTINCT h24.WALLET_DPAN)             AS DISTINCT_DPANS_L24H,
        COUNT(CASE WHEN h24.MERCHANT_COUNTRY = 'GBR' THEN 1 END) AS PURCHASES_GBR_NUM_L24H,
        -- h48 = 48-hour window
        COUNT(h48.TRANSACTION_TS)                   AS PURCHASES_NUM_L48H,
        COALESCE(SUM(h48.PURCHASE_AMOUNT), 0)       AS PURCHASES_AMT_L48H,
        -- h1wk = 7-day window
        COUNT(h1wk.TRANSACTION_TS)                  AS PURCHASES_NUM_L1WK,
        COALESCE(SUM(h1wk.PURCHASE_AMOUNT), 0)      AS PURCHASES_AMT_L1WK,
        COALESCE(MAX(h1wk.PURCHASE_AMOUNT), 0)      AS PURCHASES_MAX_L1WK,
        COUNT(DISTINCT h1wk.MERCHANT_ID)            AS DISTINCT_MERCHANTS_L1WK,
        COUNT(DISTINCT h1wk.WALLET_DPAN)            AS DISTINCT_DPANS_L1WK,

        -- Derived ratio features (same formula as scoring service — training/serving parity)
        -- COALESCE SUM numerators: DIV0(NULL, x) = NULL in Snowflake when the short window
        -- has no matches (LEFT JOIN returns no rows → SUM = NULL). Using COALESCE ensures 0.
        DIV0(COUNT(h1.TRANSACTION_TS),                COUNT(h1wk.TRANSACTION_TS))                          AS VELOCITY_RATIO_1H_L1WK,
        DIV0(COUNT(h6.TRANSACTION_TS),                COUNT(h1wk.TRANSACTION_TS))                          AS VELOCITY_RATIO_6H_L1WK,
        DIV0(COUNT(h24.TRANSACTION_TS),               COUNT(h1wk.TRANSACTION_TS))                          AS VELOCITY_RATIO_24H_L1WK,
        DIV0(COALESCE(SUM(h1.PURCHASE_AMOUNT), 0),    COALESCE(SUM(h1wk.PURCHASE_AMOUNT), 0)) ::DOUBLE     AS SPEND_BURST_1H_L1WK,
        DIV0(COALESCE(SUM(h6.PURCHASE_AMOUNT), 0),    COALESCE(SUM(h1wk.PURCHASE_AMOUNT), 0)) ::DOUBLE     AS SPEND_BURST_6H_L1WK,
        DIV0(COALESCE(SUM(h24.PURCHASE_AMOUNT), 0),   COALESCE(SUM(h1wk.PURCHASE_AMOUNT), 0)) ::DOUBLE     AS SPEND_BURST_24H_L1WK,
        COALESCE(DIV0(t.PURCHASE_AMOUNT - COALESCE(p.AVG_TXN_AMOUNT_30D, 0),
             NULLIF(COALESCE(p.AVG_TXN_AMOUNT_30D, 0), 0)), 0) ::DOUBLE                                    AS AMOUNT_PCT_DEVIATION,
        COALESCE(DIV0(t.PURCHASE_AMOUNT,
             NULLIF(COALESCE(MAX(h1wk.PURCHASE_AMOUNT), 0), 0)), 0) ::DOUBLE                                AS AMOUNT_RATIO_TO_HIST_MAX,
        COALESCE(1.0 - DIV0(COUNT(DISTINCT h24.MERCHANT_ID),
                   NULLIF(COUNT(h24.TRANSACTION_TS), 0)), 0) ::DOUBLE                                       AS MERCHANT_CONCENTRATION_24H,

        -- Customer profile (static attributes from DIM_CUSTOMERS)
        COALESCE(p.CUSTOMER_AGE, 30)             AS CUSTOMER_AGE,
        COALESCE(p.DAYS_SINCE_REGISTRATION, 365) AS DAYS_SINCE_REGISTRATION,
        COALESCE(p.IS_HIGH_RISK, 0)              AS IS_HIGH_RISK,
        COALESCE(p.LIFETIME_TXN_COUNT, 0)        AS LIFETIME_TXN_COUNT,
        COALESCE(p.LIFETIME_TXN_TOTAL, 0)        AS LIFETIME_TXN_TOTAL,
        COALESCE(p.AVG_TXN_AMOUNT_30D, 0)        AS AVG_TXN_AMOUNT_30D,
        -- Profile-derived binary flags
        CASE WHEN COALESCE(p.LIFETIME_TXN_COUNT, 0) = 0    THEN 1 ELSE 0 END AS IS_FIRST_PURCHASE,
        CASE WHEN COALESCE(p.DAYS_SINCE_REGISTRATION, 999) <= 7  THEN 1 ELSE 0 END AS IS_NEW_ACCOUNT_7D,
        CASE WHEN COALESCE(p.DAYS_SINCE_REGISTRATION, 999) <= 30 THEN 1 ELSE 0 END AS IS_NEW_ACCOUNT_30D

    FROM (SELECT * FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS SAMPLE (1000000 ROWS)) t

    -- Customer velocity: 5 self-joins, point-in-time correct
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1
        ON h1.CUSTOMER_ID = t.CUSTOMER_ID
        AND h1.TRANSACTION_TS >= DATEADD('hour', -1,  t.TRANSACTION_TS)
        AND h1.TRANSACTION_TS <  t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h6
        ON h6.CUSTOMER_ID = t.CUSTOMER_ID
        AND h6.TRANSACTION_TS >= DATEADD('hour', -6,  t.TRANSACTION_TS)
        AND h6.TRANSACTION_TS <  t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h24
        ON h24.CUSTOMER_ID = t.CUSTOMER_ID
        AND h24.TRANSACTION_TS >= DATEADD('hour', -24, t.TRANSACTION_TS)
        AND h24.TRANSACTION_TS <  t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h48
        ON h48.CUSTOMER_ID = t.CUSTOMER_ID
        AND h48.TRANSACTION_TS >= DATEADD('hour', -48, t.TRANSACTION_TS)
        AND h48.TRANSACTION_TS <  t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1wk
        ON h1wk.CUSTOMER_ID = t.CUSTOMER_ID
        AND h1wk.TRANSACTION_TS >= DATEADD('day', -7,  t.TRANSACTION_TS)
        AND h1wk.TRANSACTION_TS <  t.TRANSACTION_TS

    -- Customer profile
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE p
        ON p.CUSTOMER_ID = t.CUSTOMER_ID

    GROUP BY ALL
""")

training_df.write.save_as_table('FRAUD_DEMO_DEV.ML.TRAINING_SET', mode='overwrite')
elapsed    = time.time() - start
row_count  = session.table('FRAUD_DEMO_DEV.ML.TRAINING_SET').count()
col_count  = len(session.table('FRAUD_DEMO_DEV.ML.TRAINING_SET').columns)
print(f'Training set materialised: {row_count:,} rows, {col_count} columns, {elapsed:.1f}s')


## 3.2 Train XGBoost Model

In [ ]:
from snowflake.ml.modeling.xgboost import XGBClassifier as SnowXGBClassifier
from snowflake.snowpark.functions import avg, col as _col, iff as _iff, coalesce, lit
from snowflake.snowpark.types import (
    BooleanType, StringType, TimestampType, DateType, DoubleType
)
from sklearn.metrics import average_precision_score, roc_auc_score

training_snowdf = session.table('FRAUD_DEMO_DEV.ML.TRAINING_SET')
schema = training_snowdf.schema
print(f'Training set: {training_snowdf.count():,} rows, {len(schema.fields)} columns')

# Classify columns by type using the DataFrame schema
bool_cols = {f.name.strip('"') for f in schema.fields if isinstance(f.datatype, BooleanType)}
str_cols  = {f.name.strip('"') for f in schema.fields
             if isinstance(f.datatype, (StringType, TimestampType, DateType))}

# Exclude identifiers, target, timestamps, and string columns
EXCLUDE = {
    'CUSTOMER_ID', 'MERCHANT_ID', 'WALLET_DPAN', 'IP_ADDRESS',
    'TRANSACTION_TS', 'IS_FRAUD',
} | str_cols

feature_cols = [f.name.strip('"') for f in schema.fields
                if f.name.strip('"') not in EXCLUDE]
print(f'Features: {len(feature_cols)}')
print(f'  BOOLEAN : {sorted(bool_cols & set(feature_cols))}')
print(f'  Excluded string/time cols: {sorted(str_cols)}')

# Compute fraud rate using SQL — avoids BOOLEAN→AVG type error
fraud_rate = session.sql(
    'SELECT AVG(IFF(IS_FRAUD, 1, 0)) FROM FRAUD_DEMO_DEV.ML.TRAINING_SET'
).collect()[0][0]
scale_pos = int((1 - fraud_rate) / fraud_rate)
print(f'\nFraud rate: {fraud_rate:.4%} | scale_pos_weight: {scale_pos}')

# ── Normalize all columns to DOUBLE ──────────────────────────────────────────
# IS_FRAUD: BOOLEAN → INT (XGBoost label must be 0/1 integer)
training_snowdf = training_snowdf.with_column('IS_FRAUD', _iff(_col('IS_FRAUD'), 1, 0))

# Feature columns:
#   BOOLEAN → IFF(col, 1.0, 0.0)          Snowflake cannot CAST BOOLEAN to FLOAT
#   NUMERIC → COALESCE(col, 0.0)::DOUBLE   fills NULL ratio features with 0
for c in feature_cols:
    if c in bool_cols:
        training_snowdf = training_snowdf.with_column(
            c, _iff(_col(c), lit(1.0), lit(0.0))
        )
    else:
        training_snowdf = training_snowdf.with_column(
            c, coalesce(_col(c), lit(0.0)).cast(DoubleType())
        )

print('Column normalization complete.')

# ── Train/validation split inside Snowflake ───────────────────────────────────
train_df, val_df = training_snowdf.random_split([0.8, 0.2], seed=42)

# ── Train with Snowpark ML — XGBoost runs on the warehouse, zero data egress ──
model = SnowXGBClassifier(
    input_cols=feature_cols,
    label_cols=['IS_FRAUD'],
    output_cols=['FRAUD_PROB'],
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    tree_method='hist',
    eval_metric='aucpr',
    random_state=42,
)

print('\nTraining on warehouse (no data movement)...')
t0 = time.time()
model.fit(train_df)
print(f'Training complete in {time.time()-t0:.1f}s')

# ── Evaluate ─────────────────────────────────────────────────────────────────
proba_snowdf = model.predict_proba(val_df)

# Snowpark ML predict_proba adds new columns to the DataFrame.
# Find them by comparing against the columns before prediction.
orig_cols = set(val_df.columns)
new_cols  = [c for c in proba_snowdf.columns if c not in orig_cols]
print(f'All predict_proba output columns: {new_cols}')

# Pick the positive-class (class=1) probability column.
# Snowpark ML naming: OUTPUT_FRAUD_PROB, FRAUD_PROB_1, "FRAUD_PROB_1", etc.
# Prefer the one ending in _1; fall back to the last new column.
prob_col = next(
    (c for c in new_cols if c.strip('"').upper().endswith('_1')),
    next((c for c in new_cols if c.strip('"').upper().endswith('1')), new_cols[-1])
)
print(f'Using column for P(fraud): {prob_col}')

val_pd  = proba_snowdf.select('IS_FRAUD', prob_col).to_pandas()
auc_pr  = average_precision_score(val_pd['IS_FRAUD'].astype(int),
                                   val_pd[prob_col].astype(float))
roc_auc = roc_auc_score(val_pd['IS_FRAUD'].astype(int),
                         val_pd[prob_col].astype(float))
print(f'\nAUC-PR:  {auc_pr:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')


## 3.3 Register in Model Registry

Registers the trained model and promotes it DEV → STAGING → PROD.

In [ ]:
# Register in Snowflake Model Registry (DEV → STAGING → PROD)
# sample_input_data: use val_df (Snowpark DataFrame) — Registry accepts both
# pandas and Snowpark DataFrames for schema inference.
reg = Registry(session=session, database_name='FRAUD_DEMO_DEV', schema_name='ML')

for db in ['FRAUD_DEMO_DEV', 'FRAUD_DEMO_STAGING', 'FRAUD_DEMO_PROD']:
    try:
        session.sql(f'DROP MODEL IF EXISTS {db}.ML.FRAUD_DETECTION_MODEL').collect()
    except Exception:
        pass

model_version = reg.log_model(
    model,
    model_name='FRAUD_DETECTION_MODEL',
    version_name='v1',
    sample_input_data=val_df.select(feature_cols).limit(100),
    metrics={'auc_pr': float(auc_pr), 'roc_auc': float(roc_auc), 'scale_pos_weight': scale_pos},
    comment=f'XGBoost fraud classifier. {len(feature_cols)} features. Snowpark ML training.',
)
print(f'Registered: FRAUD_DEMO_DEV.ML.FRAUD_DETECTION_MODEL v1  AUC-PR={auc_pr:.4f}')

for db, comment in [('FRAUD_DEMO_STAGING', 'Promoted from DEV.'),
                    ('FRAUD_DEMO_PROD',    'Production model.')]:
    try:
        env_reg = Registry(session=session, database_name=db, schema_name='ML')
        env_reg.log_model(
            model, model_name='FRAUD_DETECTION_MODEL', version_name='v1',
            sample_input_data=val_df.select(feature_cols).limit(100),
            metrics={'auc_pr': float(auc_pr), 'roc_auc': float(roc_auc)},
            comment=comment,
        )
        print(f'  Promoted to {db}')
    except Exception as e:
        if 'already exist' in str(e).lower():
            print(f'  {db}: already exists — skipped')
        else:
            raise

print('\nPromotion complete: DEV -> STAGING -> PROD')


In [ ]:
# Export model for the custom SPCS scoring service.
# model._sklearn_object is the underlying xgboost.XGBClassifier fitted object.
# The SPCS container loads fraud_model.json + feature_cols.json at startup.
import json

# Extract the underlying native XGBoost model from the Snowpark ML wrapper
xgb_model = model._sklearn_object
xgb_model.save_model('/tmp/fraud_model.json')

with open('/tmp/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

session.file.put('/tmp/fraud_model.json',  '@FRAUD_DEMO_PROD.ML.MODEL_STAGE', overwrite=True, auto_compress=False)
session.file.put('/tmp/feature_cols.json', '@FRAUD_DEMO_PROD.ML.MODEL_STAGE', overwrite=True, auto_compress=False)

print('Exported to @FRAUD_DEMO_PROD.ML.MODEL_STAGE:')
print('  fraud_model.json   — XGBoost model (native JSON format)')
print(f'  feature_cols.json  — {len(feature_cols)} feature column names')
print('\nNext: nb04_serving.ipynb — build Docker image and deploy SPCS scoring service')
